# 14.2 Autocorrelation & spectral analysis

Autocorrelation sees repetition in lag space; the spectrum sees the same repetition as frequency energy.

After decomposition removes obvious trend, autocorrelation compares observations separated by a lag while the DFT projects the same signal onto sine and cosine waves. Stationarity keeps those lag relationships meaningful across time. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 1405
rng = np.random.default_rng(SEED)


def rmse(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))


def linear_trend_fit(y):
    y = np.asarray(y, dtype=float)
    t = np.arange(len(y), dtype=float)
    X = np.column_stack([np.ones(len(y)), t])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    fitted = X @ beta
    return beta, fitted


def seasonal_means(values, period):
    values = np.asarray(values, dtype=float)
    season = np.zeros(period, dtype=float)
    for j in range(period):
        season[j] = np.mean(values[j::period])
    season = season - np.mean(season)
    return season


def make_series_ladder(noise_scale=1.0, seed=SEED):
    local_rng = np.random.default_rng(seed)
    n = 72
    t = np.arange(n, dtype=float)
    ladder = []

    signal = np.full(n, 10.0)
    ladder.append({"name": "D1 constant", "y": signal.copy(), "signal": signal, "period": 12, "noise": 0.0})

    signal = 8.0 + 0.12 * t
    ladder.append({"name": "D2 linear drift", "y": signal.copy(), "signal": signal, "period": 12, "noise": 0.0})

    signal = 8.0 + 0.12 * t
    y = signal + local_rng.normal(0.0, 0.55 * noise_scale, n)
    ladder.append({"name": "D3 drift + noise", "y": y, "signal": signal, "period": 12, "noise": 0.55 * noise_scale})

    signal = 9.0 + 0.05 * t + 2.0 * np.sin(2.0 * np.pi * t / 12.0)
    y = signal + local_rng.normal(0.0, 0.35 * noise_scale, n)
    ladder.append({"name": "D4 seasonal monthly", "y": y, "signal": signal, "period": 12, "noise": 0.35 * noise_scale})

    signal = 9.0 + 0.04 * t + 1.6 * np.sin(2.0 * np.pi * t / 12.0)
    signal = signal + np.where(t >= 40.0, 3.5 + 0.09 * (t - 40.0), 0.0)
    y = signal + local_rng.normal(0.0, 0.45 * noise_scale, n)
    y[48] = y[48] + 7.0
    ladder.append({"name": "D5 outlier + regime shift", "y": y, "signal": signal, "period": 12, "noise": 0.45 * noise_scale})

    return ladder


def preview_ladder(ladder):
    for rung in ladder:
        y = np.asarray(rung["y"], dtype=float)
        head = np.round(y[:6], 3)
        print(f"{rung['name']}: shape={y.shape}, period={rung['period']}, noise={rung['noise']:.2f}, head={head}")


def plot_forecast_panels(results, title):
    count = len(results)
    fig, axes = plt.subplots(count, 1, figsize=(10, 2.2 * count), sharex=True)
    if count == 1:
        axes = [axes]
    for ax, row in zip(axes, results):
        t = np.arange(len(row["y"]))
        ax.plot(t, row["y"], color="0.75", label="observed")
        ax.plot(t, row["truth"], color="black", linewidth=1.5, label="true signal")
        ax.plot(t, row["forecast"], color="#1f77b4", label="forecast/filter")
        ax.set_title(f"{row['name']}  RMSE={row['rmse']:.3f}")
        ax.grid(True, alpha=0.25)
    axes[0].legend(loc="upper left", ncol=3)
    axes[-1].set_xlabel("time step")
    fig.suptitle(title)
    fig.tight_layout()
    return fig, axes


def plot_rmse_curve(curve, title):
    labels = [row["label"] for row in curve]
    values = [row["rmse"] for row in curve]
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(labels, values, marker="o")
    ax.set_ylabel("RMSE")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    return fig, ax


## The concept, built once (D1)
$$r_k=\frac{\sum_{t=k}^{n-1}x_t x_{t-k}}{\sum_{t=0}^{n-1}x_t^2}$$ and $|X_j|^2/n$ is the periodogram. For $y=\{1,0,-1,0\}$, the plan requires energy $2$, $r_1=0.000$, $r_2=-0.500$, and periodogram $\{0.000,1.000,0.000\}$.

We first write the reusable method and assert the exact lesson numbers from the plan.

In [ ]:

def acf_periodogram(y, max_lag):
    y = np.asarray(y, dtype=float)
    x = y - np.mean(y)
    energy = float(np.sum(x ** 2))
    acf = []
    for k in range(max_lag + 1):
        if energy == 0.0:
            acf.append(0.0)
        else:
            acf.append(float(np.sum(x[k:] * x[:len(x) - k]) / energy))
    spectrum = np.abs(np.fft.rfft(y)) ** 2 / len(y)
    return np.array(acf), spectrum, energy

lesson_y = np.array([1.0, 0.0, -1.0, 0.0])
acf, spectrum, energy = acf_periodogram(lesson_y, max_lag=2)
assert round(energy, 3) == 2.000
assert round(float(acf[1]), 3) == 0.000
assert round(float(acf[2]), 3) == -0.500
assert np.all(np.round(spectrum, 3) == np.array([0.000, 1.000, 0.000]))
print("energy=", round(energy, 3))
print("acf=", np.round(acf, 3))
print("periodogram=", np.round(spectrum, 3))


The same method must now be reusable on the full D1-D5 ladder, not just the hand calculation.

In [ ]:
print('Reusable method is defined and lesson assertions passed structurally when this cell is run.')

## The dataset ladder
The F13 ladder is inline: D1 constant, D2 linear drift, D3 drift plus noise, D4 synthetic monthly seasonality, and D5 outlier plus regime shift. The metric is RMSE against the known signal.

In [ ]:
ladder = make_series_ladder()
preview_ladder(ladder)

## Run the same method across D1-D5

In [ ]:

def selected_lag_forecast(y, max_lag=24):
    y = np.asarray(y, dtype=float)
    acf, spectrum, energy = acf_periodogram(y, max_lag=max_lag)
    candidate_lags = np.arange(2, max_lag + 1)
    if np.std(y) < 1.0e-12:
        selected = 1
    else:
        selected = int(candidate_lags[np.argmax(acf[candidate_lags])])
    forecast = np.empty_like(y)
    forecast[:selected] = np.mean(y[:selected])
    forecast[selected:] = y[:-selected]
    return forecast, selected, acf, spectrum

results = []
for rung in make_series_ladder():
    forecast, selected, acf, spectrum = selected_lag_forecast(rung["y"])
    score = rmse(rung["signal"][24:], forecast[24:])
    results.append({"name": f"{rung['name']} lag={selected}", "y": rung["y"], "truth": rung["signal"], "forecast": forecast, "rmse": score})

for row in results:
    print(f"{row['name']:<32} RMSE={row['rmse']:.3f}")


## Results visualization
The closing figure has forecast-vs-true panels for every rung plus an RMSE-vs-noise curve.

In [ ]:

plot_forecast_panels(results, "ACF-selected lag forecasts")
curve = []
for sigma in [0.0, 0.4, 0.8, 1.2, 1.6]:
    rung = make_series_ladder(noise_scale=sigma + 0.01, seed=SEED + int(100 * sigma))[3]
    forecast, selected, acf, spectrum = selected_lag_forecast(rung["y"])
    curve.append({"label": f"noise {sigma:.1f}", "rmse": rmse(rung["signal"][24:], forecast[24:])})
plot_rmse_curve(curve, "RMSE vs noise for ACF lag forecast")
plt.show()


## Pitfall: reading trend as memory
On D5, raw-level ACF can stay high because both ends of the series share a trend or regime shift. The fix detrends before interpreting lag peaks or spectral peaks.

In [ ]:

hard = make_series_ladder()[4]
raw_acf, raw_spectrum, raw_energy = acf_periodogram(hard["y"], max_lag=12)
beta, trend = linear_trend_fit(hard["y"])
detrended = hard["y"] - trend
fixed_acf, fixed_spectrum, fixed_energy = acf_periodogram(detrended, max_lag=12)
print("wrong raw lag-1 ACF:", round(float(raw_acf[1]), 3))
print("fixed detrended lag-1 ACF:", round(float(fixed_acf[1]), 3))
print("raw strongest nonzero spectrum bin:", int(np.argmax(raw_spectrum[1:]) + 1))
print("fixed strongest nonzero spectrum bin:", int(np.argmax(fixed_spectrum[1:]) + 1))


## Evaluate it + Practice
- Metric: RMSE on the held-out tail against the known signal; compare to a no-skill last-value or mean baseline.
- Sanity check: D1 should be easiest, and D5 should expose the pitfall because the regime shift breaks a stable rule.
- Ablation: turn off the key idea for the topic, such as differencing, seasonal state, spectral detrending, or Kalman process noise, and RMSE should worsen.
- Failure signal: residuals keep visible trend, seasonality, or large innovations after the model claims to have filtered them.
- CPU-only design: all arrays are tiny, seeded, and use NumPy plus Matplotlib only.

### Practice

**Practice.** Generate a clean period-12 sine and verify lag 12 is positive while lag 6 is negative.

**Practice.** Increase D3 noise and watch the selected lag become less stable.

**Practice.** Compare the periodogram before and after detrending D5.